# Data & Graph Construction

In [23]:
# ============================================
# 1. Data & Graph Construction
# ============================================

# List of all nodes in the route planning problem
nodes = [
    "SunU",
    "Vidhipriya",
    "Wen Li",
    "Keertana",
    "Hui San",
    "Qi Yung"
]


# Raw edge data:
# Format: (From, To, Normalised Carbon Emission, Normalised Travel Expense)
raw_edges = [
    ("SunU", "Vidhipriya", 0.0000, 0.0000),
    ("SunU", "Wen Li", 0.1625, 0.2700),
    ("SunU", "Keertana", 0.0677, 0.4800),

    ("Vidhipriya", "Hui San", 0.2604, 0.3300),
    ("Vidhipriya", "Wen Li", 0.1042, 0.1000),
    ("Vidhipriya", "Keertana", 0.1313, 0.6000),

    ("Wen Li", "Hui San", 0.1771, 0.2000),

    ("Keertana", "Hui San", 0.2281, 0.4900),
    ("Keertana", "Qi Yung", 0.1615, 1.0000),

    ("Hui San", "Qi Yung", 0.2396, 0.5400)
]


# Initialise empty adjacency list for each node
graph = {
    node: []
    for node in nodes
}


# Construct an undirected graph
# Each connection is added in both directions
for source, destination, carbon, expense in raw_edges:

    # Calculate weighted path cost
    # Cost = 0.6(Carbon Emission) + 0.4(Travel Expense)
    cost = (0.6 * carbon) + (0.4 * expense)

    # Add forward and reverse edges
    graph[source].append(
        (destination, cost)
    )

    graph[destination].append(
        (source, cost)
    )


# Display adjacency list with costs rounded to 5 decimal places
print("Graph Adjacency List:\n")

for node in graph:
    formatted_edges = [
        (neighbour, round(cost, 5))
        for neighbour, cost in graph[node]
    ]

    print(f"{node}: {formatted_edges}")

Graph Adjacency List:

SunU: [('Vidhipriya', 0.0), ('Wen Li', 0.2055), ('Keertana', 0.23262)]
Vidhipriya: [('SunU', 0.0), ('Hui San', 0.28824), ('Wen Li', 0.10252), ('Keertana', 0.31878)]
Wen Li: [('SunU', 0.2055), ('Vidhipriya', 0.10252), ('Hui San', 0.18626)]
Keertana: [('SunU', 0.23262), ('Vidhipriya', 0.31878), ('Hui San', 0.33286), ('Qi Yung', 0.4969)]
Hui San: [('Vidhipriya', 0.28824), ('Wen Li', 0.18626), ('Keertana', 0.33286), ('Qi Yung', 0.35976)]
Qi Yung: [('Keertana', 0.4969), ('Hui San', 0.35976)]


# State Representation & Data Structure Design

In [24]:
# ============================================
# 2. State Representation & Data Structure Design
# ============================================

# Define the state structure used by A* Search.
# Each state stores the current location, visited nodes,
# accumulated cost g(n), and the route taken so far.

class State:

    def __init__(self, current_node, visited_set, g_cost, path):

        # Current node being explored
        self.current_node = current_node

        # Nodes that have already been visited
        self.visited_set = visited_set

        # Accumulated path cost from the starting node
        self.g_cost = g_cost

        # List of nodes representing the route history
        self.path = path


    # Display state information in readable format
    def __repr__(self):

        return (
            f"State("
            f"Node={self.current_node}, "
            f"Visited={self.visited_set}, "
            f"g(n)={self.g_cost:.5f}, "
            f"Path={self.path})"
        )


# Create the initial state
# Starting point: Sunway University

initial_state = State(
    current_node="SunU",
    visited_set={"SunU"},
    g_cost=0,
    path=["SunU"]
)


# Display initial state
print(initial_state)

State(Node=SunU, Visited={'SunU'}, g(n)=0.00000, Path=['SunU'])


# Heuristic Function Design

In [22]:
# ============================================
# 3. Heuristic Function Design
# ============================================

# Define the heuristic function h(n).
# The heuristic estimates the remaining cost by finding the minimum cost to an unvisited neighbouring node.

def heuristic(node, visited):

    # Store possible costs to unvisited neighbours
    possible_costs = []


    # Check all neighbouring nodes
    for neighbour, cost in graph[node]:

        # Ignore nodes that have already been visited
        if neighbour not in visited:
            possible_costs.append(cost)


    # If all connected nodes have been visited,no remaining cost is required
    if not possible_costs:
        return 0


    # Return the minimum estimated remaining cost
    return min(possible_costs)



# Test heuristic calculation for the starting node

print(
    "h(SunU):",
    round(
        heuristic("SunU", {"SunU"}),
        5
    )
)

h(SunU): 0.0


# A* Search Core Engine with Goal Test

In [21]:
# ============================================
# 4. A* Search Core Engine
# ============================================

import heapq


def a_star_search():

    # Open list stores unexplored states using priority queue
    # States with the lowest f(n) are expanded first
    open_list = []

    # Closed list stores already explored states
    closed_list = set()


    # Create initial state at Sunway University
    start = State(
        current_node="SunU",
        visited_set={"SunU"},
        g_cost=0,
        path=["SunU"]
    )


    # Calculate initial f(n) = g(n) + h(n)
    f_start = (
        start.g_cost +
        heuristic(
            start.current_node,
            start.visited_set
        )
    )


    # Add starting state into Open List
    heapq.heappush(
        open_list,
        (f_start, start)
    )


    # Continue searching until Open List is empty
    while open_list:

        # Select state with the lowest f(n)
        current_f, current = heapq.heappop(
            open_list
        )


        # Create unique state identifier
        # based on current location and visited nodes
        state_key = (
            current.current_node,
            tuple(sorted(current.visited_set))
        )


        # Skip states that have already been explored
        if state_key in closed_list:
            continue


        # Add current state into Closed List
        closed_list.add(state_key)


        # Goal test:Stop when all locations have been visited
        if current.visited_set == set(nodes):
            return current


        # Explore neighbouring nodes
        for neighbour, cost in graph[current.current_node]:


            # Prevent revisiting previously visited nodes
            if neighbour in current.visited_set:
                continue


            # Create updated visited set
            new_visited = current.visited_set.copy()
            new_visited.add(neighbour)


            # Create updated route path
            new_path = current.path.copy()
            new_path.append(neighbour)


            # Calculate new accumulated cost g(n)
            new_g = current.g_cost + cost


            # Create new search state
            new_state = State(
                current_node=neighbour,
                visited_set=new_visited,
                g_cost=new_g,
                path=new_path
            )


            # Calculate f(n) = g(n) + h(n)
            f_cost = (
                new_g +
                heuristic(
                    neighbour,
                    new_visited
                )
            )


            # Add new state into Open List
            heapq.heappush(
                open_list,
                (f_cost, new_state)
            )


    # Return None if no valid route is found
    return None

# Output & Integration

In [20]:
# ============================================
# 5. Output & Integration
# ============================================

# Display the final A* search result
def display_result(result):

    # Check whether a valid route was found
    if result is None:

        print("No route found")


    else:

        # Display final route
        print("Optimal Route:")
        print(
            " → ".join(result.path)
        )


        # Display all visited locations
        print("\nVisited Nodes:")
        print(
            result.visited_set
        )


        # Display total accumulated path cost
        print("\nTotal Cost:")
        print(
            round(
                result.g_cost,
                5
            )
        )



# Main program execution
# Run A* Search and display the result

result = a_star_search()

display_result(result)

Optimal Route:
SunU → Vidhipriya → Wen Li → Hui San → Keertana → Qi Yung

Visited Nodes:
{'Wen Li', 'Qi Yung', 'SunU', 'Vidhipriya', 'Hui San', 'Keertana'}

Total Cost:
1.11854
